In [19]:
from sentence_transformers import SentenceTransformer
import pandas as pd 
import numpy as np
import re 
from src.LLM.embedding import model,category_embeddings,categories
from src.LLM.category_select import get_top_categories
from src.Search.Data_ingest import data_ingest 



In [20]:

products,categories = data_ingest()

Ingesting Data
Procesisng Data
Columns:
['Unnamed: 0', 'asin', 'title', 'category_name', 'category_id', 'stars', 'reviews', 'price', 'price_inr', 'imgUrl', 'productURL']

Number of products: 1393565


In [4]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
client_2=Groq(api_key=os.getenv("GROQ_API_KEY_2"))

# Models list for further changing 
# models = client.models.list()
# for model in models.data:
#     print(model.id)


In [5]:
from src.LLM.TEMPLATES.main_template import PROMPT_TEMPLATE 
from src.LLM.TEMPLATES.min_attribute import PROMPT_TEMPLATE_ATT_MIN 
from src.LLM.TEMPLATES.product_template import PROMPT_TEMPLATE_PRODUCT

In [ ]:
import json 

def attributes(template,promt_main):
    att_template={}
    att_template["user_promt"]=promt_main 
    att_template["items"]=[]
    for i,values in enumerate(template['items']):
        att_template["items"].append({})
        att_template["items"][i]['name']=values['name']
        att_template["items"][i]['preferences']=values['preferences']

    prompt =  PROMPT_TEMPLATE_ATT_MIN.replace(
        "{input_json}",
        json.dumps(att_template, ensure_ascii=False)
    )
    response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": "Extract attributes accurately. Return JSON only."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
    )
    att_template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    for i,values in enumerate(template['items']):
        template["items"][i]['attributes']=att_template["items"][i]['attributes']
    return template

def elaborate_product(product_name,preferences):
    input_data = {"item": product_name,"preferences":preferences}

    prompt = PROMPT_TEMPLATE_PRODUCT.replace(
        "{input_json}",
        json.dumps(input_data, ensure_ascii=False)
    )

    response = client_2.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Elaborate product names for semantic search. "
                    "Return JSON only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    print("Elaborate Promt:",result["output"])
    return result["output"]
     

    
def adding_category(template,top_k):
    llm_items=template["items"]
    for i in range(len(llm_items)):
            llm_category=template["items"][i]["name"]
            preferences=template["items"][i]["preferences"]
            elaborate_promt=elaborate_product(llm_category,preferences)
            results,category_ids=get_top_categories(elaborate_promt,top_k)
            template["items"][i]["category_id"]=category_ids 
    return template

def extract_requirements(user_prompt):
    prompt = PROMPT_TEMPLATE.replace("{user_prompt}", user_prompt)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "Extract requirements accurately. Return JSON only."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    template=attributes(template,user_prompt)

    #top Categories
    top_k=3
    main_template=adding_category(template,top_k)
    
    
    return main_template

In [7]:
prompt="i want red shoes and black shoes"
main_template=extract_requirements(prompt)
print(main_template)

Elaborate Promt: Red shoes are footwear designed for providing comfort and style during everyday wear.
Top 3 categories:
Category_id          Category_name
122 Women's Shoes
114 Men's Shoes
97 Girls' Shoes
Elaborate Promt: Black shoes are footwear designed for wearing on the feet.
Top 3 categories:
Category_id          Category_name
114 Men's Shoes
122 Women's Shoes
97 Girls' Shoes
{'items': [{'name': 'Shoes', 'keywords': ['shoes'], 'preferences': ['red'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['red']}, 'category_id': [122, 114, 97]}, {'name': 'Shoes', 'keywords': ['shoes'], 'preferences': ['black'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['black']}, 'category_id': [114, 122, 97]}]}


In [8]:
main_template

{'items': [{'name': 'Shoes',
   'keywords': ['shoes'],
   'preferences': ['red'],
   'budget': None,
   'min_rating': None,
   'min_reviews': None,
   'attributes': {'color': ['red']},
   'category_id': [122, 114, 97]},
  {'name': 'Shoes',
   'keywords': ['shoes'],
   'preferences': ['black'],
   'budget': None,
   'min_rating': None,
   'min_reviews': None,
   'attributes': {'color': ['black']},
   'category_id': [114, 122, 97]}]}

In [10]:
from src.Search.search import Searching

obj=Searching(products,categories)

Getting all the Data


In [11]:
# prompt="show me black hiking shoes under 50 with white t-shirt with  L size with black shorts with L size"
# prompt=str(input("Enter the prompt"))
# prompt="Show me one black shirt for boys and one blue shirts for girls"
# print(prompt)
# main_template=extract_requirements(prompt)
# print(main_template)
solutions=obj.finding_products(main_template)

{'items': [{'name': 'Shoes', 'keywords': ['shoes'], 'preferences': ['red'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['red']}, 'category_id': [122, 114, 97]}, {'name': 'Shoes', 'keywords': ['shoes'], 'preferences': ['black'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['black']}, 'category_id': [114, 122, 97]}]}
shoes-1 domain size: 15842
shoes-2 domain size: 15842
Total Possible Combiantions: 250968964

Solution 1
TOTAL SOLUTIONS: 100
Item: shoes-1
Unique ID: B0C733HZHQ
Product: Men Dress Shoe Prom Slip on Loafer and Lace Up Cap Toe Oxford Modern Metallic Glitter for Wedding Prom Black Blue Gold Pink Pewter Silver Red
Price: 5699.05
Category ID: 114
Category Name: Men's Shoes
------------------------
Item: shoes-2
Unique ID: B0BN5X2ZK4
Product: Non Slip Work Shoes for Men Kitchen Chef Slip Resistant Shoe Waterproof Food Service Restaurant Slip on Sneakers Walking and Casual Air Cushion Working Footwear Black

In [ ]:

from src.utils.display import display_bundles

display_bundles(solutions)